# Benchmark WASB tennis ball detectors
Runs the published WASB-SBDT evaluation (BMVC 2023) on the TrackNet tennis dataset: WASB, TrackNetV2, ResTrackNetV2 and MonoTrack, each with its tennis weights. Upstream has not released training code, so this notebook evaluates and does not train.

The split is upstream's: game1–game7 train, game8–game10 test, grouped by match. Treat game8–game10 as held-out. Do not tune thresholds on them. Upstream code and weights are MIT. The dataset licence has not been verified, so keep data, weights and outputs out of the repository and use them for local research only.

The first run downloads the upstream tennis weights and `Dataset.zip` (2.56 GB) into `MyDrive/TennisAI-external/` and reuses them after that. Results and logs are written to `TennisAI-external/wasb_results/`.

## 1. Setup
Select **Runtime → Change runtime type → GPU**.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
from google.colab import drive
drive.mount('/content/drive')
EXT = Path('/content/drive/MyDrive/TennisAI-external')
WEIGHTS, RESULTS = EXT/'wasb_weights', EXT/'wasb_results'
RESULTS.mkdir(parents=True, exist_ok=True)
WEIGHTS.mkdir(parents=True, exist_ok=True)
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive free: {free/1e9:.1f} GB')

## 2. Tennis weights
Google Drive IDs from upstream `MODEL_ZOO.md` at the pinned commit. Files already in Drive are reused.

In [ ]:
import gdown
WEIGHT_IDS = {
    'wasb': '14AeyIOCQ2UaQmbZLNQJa1H_eSwxUXk7z',
    'tracknetv2': '1BCnmvDX-LZpbkk4vlMEXMm-uzCoqJzDx',
    'restracknetv2': '112qHZpPWgqCeZbbFXC0oRulrVsokzWYZ',
    'monotrack': '1mC3yWf6ySlzF-1d-s_LNH1QWexHz2M3D',
}
for model, file_id in WEIGHT_IDS.items():
    target = WEIGHTS/f'{model}_tennis_best.pth.tar'
    if not target.exists():
        gdown.download(id=file_id, output=str(target), quiet=True)
    print(target.name, f'{target.stat().st_size/1e6:.1f} MB')

## 3. Tennis dataset
The upstream SharePoint link needs its share cookie before the file URL resolves, so `curl` keeps a cookie jar. The archive is cached in Drive and unpacked to the faster local disk.

In [ ]:
SHARE = 'https://nycu1-my.sharepoint.com/:u:/g/personal/tik_m365_nycu_edu_tw/ETCr6-M0e1VDhGCdMbvljcsBu31AJTO5xa_1cW8pHa7niA?e=55tLJ9'
ZIP, ZIP_BYTES = EXT/'Dataset.zip', 2_564_111_322
if not ZIP.exists() or ZIP.stat().st_size != ZIP_BYTES:
    jar, part = '/content/sharepoint.jar', Path('/content/Dataset.zip.part')
    subprocess.run(['curl', '-sSL', '-c', jar, '-b', jar, '-o', '/dev/null', SHARE], check=True)
    subprocess.run(['curl', '-SL', '--fail', '-c', jar, '-b', jar, '-o', str(part), SHARE + '&download=1'], check=True)
    assert part.stat().st_size == ZIP_BYTES, part.stat().st_size
    shutil.copy2(part, ZIP); part.unlink()
DATA = Path('/content/datasets')
if not (DATA/'tennis'/'game10').exists():
    DATA.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', str(ZIP), '-d', str(DATA)], check=True)
    (DATA/'Dataset').rename(DATA/'tennis')
print(sorted(p.name for p in (DATA/'tennis').iterdir()))

## 4. WASB-SBDT code
Pinned to the upstream commit that was reviewed. Colab's current Python and PyTorch replace the Dockerfile's versions, which needs two compatibility changes. Hydra 1.2 fails on Python 3.11+, so this uses 1.3.2. NumPy 2 removed `np.Inf`, so the clone is patched to `np.inf`. Neither change affects the metrics.

In [ ]:
import re
WASB_COMMIT = '923462cacdeb3353b84ddebdedb3f4b7a8553b0f'
HYDRA = 'hydra-core==1.3.2'
WASB = Path('/content/WASB-SBDT')
if not WASB.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/nttcom/WASB-SBDT', str(WASB)], check=True)
subprocess.run(['git', '-C', str(WASB), 'checkout', '-q', '--force', WASB_COMMIT], check=True)
PATCHED = []
for source in sorted((WASB/'src').rglob('*.py')):
    text = source.read_text()
    if re.search(r'np\.Inf\b', text):
        source.write_text(re.sub(r'np\.Inf\b', 'np.inf', text))
        PATCHED.append(str(source.relative_to(WASB)))
print('np.Inf -> np.inf in', PATCHED)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', HYDRA, 'einops', 'timm'], check=True)
(WASB/'pretrained_weights').mkdir(exist_ok=True)
for weight in WEIGHTS.glob('*_tennis_best.pth.tar'):
    shutil.copy2(weight, WASB/'pretrained_weights'/weight.name)

## 5. Evaluate on the held-out test matches
Upstream evaluation settings: score threshold 0.5 and a 4-pixel match distance. Each model's full log is saved to Drive.

In [ ]:
MODELS = {
    'wasb': [],
    'tracknetv2': ['detector.postprocessor.use_hm_weight=False', 'tracker=intra_frame_peak'],
    'restracknetv2': ['detector.postprocessor.use_hm_weight=False', 'tracker=intra_frame_peak'],
    'monotrack': ['detector.postprocessor.use_hm_weight=False', 'tracker=intra_frame_peak'],
}
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()
def total_metrics(log):
    # The last two table rows after '-- TOTAL --' are the header and the all-test-match values.
    lines = log[log.rfind('-- TOTAL --'):].splitlines()
    rows = [[cell.strip() for cell in line.split(' - ', 1)[-1].strip('| ').split('|')]
            for line in lines if 'utils.evaluator' in line and '|' in line and '----' not in line]
    return dict(zip(*rows[-2:])) if len(rows) >= 2 else None
runs = {}
for model, extra in MODELS.items():
    weight = WASB/'pretrained_weights'/f'{model}_tennis_best.pth.tar'
    out = RESULTS/model
    command = [sys.executable, 'main.py', '--config-name=eval', 'dataset=tennis', f'model={model}',
               f'dataset.root_dir={DATA/"tennis"}', 'runner.gpus=[0]', f'output_dir={out}',
               f'detector.model_path={weight}', *extra]
    print('$', ' '.join(command))
    result = subprocess.run(command, cwd=WASB/'src', capture_output=True, text=True)
    out.mkdir(parents=True, exist_ok=True)
    (out/'stdout.log').write_text(result.stdout + result.stderr)
    print('\n'.join((result.stdout + result.stderr).strip().splitlines()[-15:]))
    runs[model] = {'returncode': result.returncode, 'test_total': total_metrics(result.stdout + result.stderr),
                   'weights_sha256': sha256(weight), 'command': command[1:]}

## 6. Provenance
The summary records the inputs and each model's metrics over all test matches. Full logs are in `stdout.log`. These numbers come from broadcast footage only and are not evidence of accuracy on phone video.

In [ ]:
summary = {
    'label': 'development-only benchmark; not a promotion gate',
    'wasb_commit': WASB_COMMIT,
    'compatibility': {'hydra': HYDRA, 'np_inf_patched': PATCHED},
    'weight_drive_ids': WEIGHT_IDS,
    'dataset': {'archive': 'Dataset.zip', 'bytes': ZIP.stat().st_size, 'sha256': sha256(ZIP), 'test_matches': ['game8', 'game9', 'game10']},
    'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
    'runs': runs,
}
(RESULTS/'summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))